# Lab 4 · Fase 4 en Kaggle (notebook reducido)

Solo las celdas imprescindibles del notebook de César (rama `CesarV1`) y los experimentos de la fase 4 sobre `p2_07`.
1. *Add Input* → el dataset privado con `db.zip`. *Accelerator* → **GPU T4 x2**.
2. **Save Version → Save & Run All** para que corra en segundo plano.
3. Al terminar: pestaña *Output* → `outputs/Lab_4_2026_2027/search/` (`results_p4_m*.csv`, `p4_*_epochs.csv`, `p4_summary.csv`, `.pt`).

In [ ]:
%matplotlib inline

In [ ]:
import os
from pathlib import Path
import copy
import numbers
import random
import time
import zipfile

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt

from PIL import Image
from skimage import io, transform, util

import torch  
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, utils, models


# Install missing dependencies in your chosen environment before running.
# In a notebook, you can use:
# %pip install torch torchvision pandas scikit-image matplotlib scikit-learn pillow

In [ ]:
SEED = 42
CLASS_NAMES = ['Water', 'Very Low', 'Low', 'Moderate', 'High', 'Very High', 'Non-burnable']
NUM_CLASSES = len(CLASS_NAMES)
MAX_TRAIN_SAMPLES = 0  # Set to 0 to use the full training split. 500 is a good number for quick experiments
# Windows cannot use worker processes with notebook-defined classes -> 0. Linux: up to 4 workers.
NUM_WORKERS = 0 if os.name == 'nt' else min(4, os.cpu_count() or 1)
# Epoch: num. of iterations over the entire dataset over batches.
# Between those iterations the model weights are updated based on the loss function and optimizer.
CUSTOM_EPOCHS = 30
FT_EPOCHS = 5
# LERNING RATE: Step Size of the Optimizer at each iteration while moving toward a minimum of a loss function. Affects how weights are updated
CUSTOM_LR = 1e-2
FT_LR = 1e-3

# Note: PIXEL_MEAN and PIXEL_STD come directly from AlexNet's
# documentation in PyTorch:
# https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.alexnet.html
# They are the per-channel (R, G, B) mean and standard deviation of the ImageNet training set,
# the dataset the pretrained torchvision models (AlexNet, etc.) were trained on. Normalising
# with them centres each channel around 0 with unit variance (stable, faster training) and,
# for the fine-tuned network, gives it the input statistics it was pretrained with.
# For a network trained from scratch they are not mandatory (the dataset's own mean/std
# could be used instead), but they are a reasonable default.
PIXEL_MEAN = [0.485, 0.456, 0.406]
PIXEL_STD = [0.229, 0.224, 0.225]

DATA_ARCHIVE = Path('db.zip')
DATA_ROOT = Path('data')
DB_DIR = DATA_ROOT / 'db'
OUTPUT_DIR = Path('outputs') / 'Lab_4_2026_2027'

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

seed_everything()
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'Class order: {CLASS_NAMES}')

In [ ]:
# ---- Kaggle setup: locate the dataset added with "Add Input" (zip or already extracted by Kaggle) ----
import shutil, sys
IN_COLAB = False
IN_KAGGLE = Path('/kaggle/input').exists()
WORK = Path('/kaggle/working') if IN_KAGGLE else Path('.')
if IN_KAGGLE:
    zips = list(Path('/kaggle/input').rglob('db.zip'))
    csvs = list(Path('/kaggle/input').rglob('train_reduced.csv'))
    if zips:                                   # uploaded as a zip that Kaggle kept as-is -> extract locally
        DATA_ARCHIVE = WORK / 'db.zip'
        if not DATA_ARCHIVE.exists():
            shutil.copy(zips[0], DATA_ARCHIVE)
        DATA_ROOT = WORK / 'data'
        DB_DIR = DATA_ROOT / 'db'
    elif csvs:                                 # Kaggle extracted the zip on upload -> read it in place
        DB_DIR = csvs[0].parent
        DATA_ARCHIVE = Path('/nonexistent.zip')   # skip the extraction cell
    else:
        raise FileNotFoundError('Add the FireRisk dataset with "Add Input" (db.zip or its extracted db/ folder).')
print('DB_DIR =', DB_DIR)

In [ ]:
required_csvs = [DB_DIR / f'{split}_reduced.csv'
                 for split in ('train', 'val', 'test')]

if DATA_ARCHIVE.is_file():
    DATA_ROOT.mkdir(parents=True, exist_ok=True)
    restored_files = 0
    with zipfile.ZipFile(DATA_ARCHIVE) as archive:

        for member in archive.infolist():
            if not member.filename.startswith('db/'):
                continue
            target = DATA_ROOT / member.filename
            if not member.is_dir() and target.is_file() and target.stat().st_size == member.file_size:
                continue
            if not target.resolve().is_relative_to(DATA_ROOT.resolve()):
                raise ValueError('The archive contains an invalid dataset path.')
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
            elif not target.is_file() or target.stat().st_size != member.file_size:
                archive.extract(member, DATA_ROOT)
                restored_files += 1
    print(f'Dataset ready at {DB_DIR.resolve()}; restored {restored_files} files.')

if not all(path.is_file() for path in required_csvs):
    raise FileNotFoundError(
        'Dataset CSV files were not found. Start the kernel in the notebook directory '
        'or update DATA_ARCHIVE and DATA_ROOT in the setup cell.')

for csv_path in required_csvs:
    ids = pd.read_csv(csv_path, usecols=['id'], dtype={'id': str})['id']
    for image_id in ids:
        image_path = DB_DIR / 'images' / f'{image_id}.jpg'
        if not image_path.is_file() or image_path.stat().st_size == 0:
            raise FileNotFoundError(
                f'Missing or empty image: {image_path}. Supply the original db.zip '
                'and rerun this setup cell to restore it.')

print('All indexed images exist and are nonempty.')

In [ ]:
class FireRiskDataset(Dataset):
    """Read RGB images and integer labels using a CSV index.

    Args:
        csv_file: CSV with columns 'id' and 'label'.
        root_dir: Dataset directory containing the 'images' subdirectory.
        transform: Optional callable that receives and returns a sample dictionary.
        maxSize: Maximum subset size; 0 selects the full split.
    """

    # Creates and checks the dataset
    def __init__(self, csv_file, root_dir, transform=None, maxSize=0, preload=False):

        if not isinstance(maxSize, numbers.Integral) or maxSize < 0:
            raise ValueError('maxSize must be a nonnegative integer.')

        self.dataset = pd.read_csv(csv_file, dtype={'id': str, 'label': int})

        if not {'id', 'label'}.issubset(self.dataset.columns) or self.dataset.empty:
            raise ValueError('The CSV must contain nonempty id and label columns.')
        if not self.dataset['label'].isin(range(-1, NUM_CLASSES)).all():
            raise ValueError('Labels must be -1 (hidden test labels) or 0 through 6.')
        if maxSize > 0:
            indices = np.random.RandomState(SEED).permutation(len(self.dataset)) # shuffle the dataset
            self.dataset = self.dataset.iloc[indices[:maxSize]].reset_index(drop=True) # after shuffling, select 1st maxSize rows

        self.img_dir = Path(root_dir) / 'images'
        self.transform = transform
        self.classes = CLASS_NAMES.copy()

        # DECODE IMG in RAM to SPEED UP TRAINING. Optional (default preload is False and wouldn't be used if memory is limited)
        self.cache = None
        if preload:
            first = self._read(0)
             # Array of size (len(dataset), H, W, 3) to store all images in RAM:
            self.cache = np.empty((len(self.dataset), *first.shape), dtype=np.uint8)
            for i in range(len(self.dataset)):
                self.cache[i] = self._read(i) # Store each image in the cache

    def _read(self, idx):
        """Read one image FROM DISK as an H x W x 3 uint8 array"""
        image = io.imread(self.img_dir / f"{self.dataset.iloc[int(idx)]['id']}.jpg")
        if image.ndim == 2:
            image = np.repeat(image[..., None], 3, axis=2) # Convert grayscale to RGB
        elif image.ndim == 3 and image.shape[2] == 4:
            image = image[..., :3] # Discard alpha channel if present
        if image.ndim != 3 or image.shape[2] != 3: 
            raise ValueError('Expected an RGB image with three channels.')
        return image


    def __len__(self):
        return len(self.dataset)

    # Get a sample from the dataset. Each sample is a dictionary: {'image': image, 'label': label}.
    # When doing dataset[i], the __getitem__ method is called, where i is the index and ['image'] is the image
    def __getitem__(self, idx):

        if torch.is_tensor(idx): 
            idx = idx.item()
        idx = int(idx)
        raw = self.cache[idx] if self.cache is not None else self._read(idx) # Read from cache if available
        sample = {'image': util.img_as_float32(raw), 'label': int(self.dataset.iloc[idx]['label'])}

        # APPLY TRANSFORMATIONS (if any) to the sample before returning it
        return self.transform(sample) if self.transform is not None else sample

In [ ]:
def _crop_size(output_size):
    # Returns a tuple representing the crop size

    if isinstance(output_size, numbers.Integral):
        output_size = (int(output_size), int(output_size)) # Cropped image is square if only one integer is provided

    if (not isinstance(output_size, tuple) or len(output_size) != 2 or
        any(not isinstance(v, numbers.Integral) or v <= 0 for v in output_size)):
        raise ValueError('Size must be a positive integer or a pair of positive integers.')

    return tuple(int(v) for v in output_size)

class Rescale:
    """Resize to (height, width), or resize the shorter side preserving aspect ratio."""
    # If output_size is an integer, resize calculating the scale of that integer with respect to the short side

    def __init__(self, output_size):
        self.output_size = (int(output_size) if isinstance(output_size, numbers.Integral)
                            else _crop_size(output_size))
        if isinstance(self.output_size, int) and self.output_size <= 0:
            raise ValueError('The output size must be positive.')

    def __call__(self, sample):
        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        if isinstance(self.output_size, int):
            scale = self.output_size / min(h, w)
            new_h, new_w = round(h * scale), round(w * scale)
        else:
            new_h, new_w = self.output_size
        image = transform.resize(image, (new_h, new_w), anti_aliasing=True)

        return {'image': image.astype(np.float32), 'label': label}

class RandomCrop:
    """Choose uniformly among all valid crops, including the final row/column."""

    def __init__(self, output_size):
        self.output_size = _crop_size(output_size)

    def __call__(self, sample):

        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if new_h > h or new_w > w:
            raise ValueError('Crop exceeds image dimensions; resize the image first.')
        # Choose startring coordinates for the crop randomly within the valid range to avoid going out of bounds
        top = np.random.randint(0, h - new_h + 1)
        left = np.random.randint(0, w - new_w + 1)

        return {'image': image[top:top + new_h, left:left + new_w], 'label': label}

class CenterCrop:
    """Take a deterministic central crop without silently changing its size."""

    def __init__(self, output_size):
        self.output_size = _crop_size(output_size)

    def __call__(self, sample):

        image, label = sample['image'], sample['label']
        h, w = image.shape[:2]
        new_h, new_w = self.output_size

        if new_h > h or new_w > w:
            raise ValueError('Crop exceeds image dimensions; resize the image first.')
        top, left = (h - new_h) // 2, (w - new_w) // 2

        return {'image': image[top:top + new_h, left:left + new_w], 'label': label}

class ToTensor:
    """Convert HWC images to contiguous float32 CHW tensors and labels to int64."""

    def __call__(self, sample):
        image = np.ascontiguousarray(sample['image'].transpose(2, 0, 1), dtype=np.float32)
        return {'image': torch.from_numpy(image), # Numpy array converted to PyTorch tensor
                'label': torch.tensor(sample['label'], dtype=torch.long)} # Label converted to PyTorch tensor

class Normalize:
    """Apply per-channel normalisation without mutating the input tensor."""

    def __init__(self, mean, std):

        self.mean = np.asarray(mean, dtype=np.float32)
        self.std = np.asarray(std, dtype=np.float32)

        if (self.mean.ndim != 1 or self.std.shape != self.mean.shape or
            not np.isfinite(self.mean).all() or not np.isfinite(self.std).all() or
            not (self.std > 0).all()):
            raise ValueError('Mean/std must be matching finite vectors with positive std.')

    def __call__(self, sample):

        image = sample['image']

        if image.ndim != 3 or image.shape[0] != len(self.mean):
            raise ValueError('The CHW tensor and normalisation vectors must match.')

        mean = torch.as_tensor(self.mean, dtype=image.dtype, device=image.device)[:, None, None] # convert mean to tensor
        std = torch.as_tensor(self.std, dtype=image.dtype, device=image.device)[:, None, None]
        return {'image': (image - mean) / std, 'label': sample['label']}

In [ ]:
class GPUAugment:
    """Random augmentation + normalisation of a batch (N x 3 x H x W, floats in [0, 1]) on the GPU."""
    def __init__(self, mean, std, gamma=0.1, brightness=0.05, contrast=0.05, saturation=0.05):
        self.mean = torch.tensor(mean).view(1, 3, 1, 1)
        self.std = torch.tensor(std).view(1, 3, 1, 1)
        self.gamma, self.brightness, self.contrast, self.saturation = gamma, brightness, contrast, saturation

    @staticmethod
    def _factor(n, amount, device):
        return 1 + (torch.rand(n, 1, 1, 1, device=device) * 2 - 1) * amount

    @staticmethod
    def _gray(x):
        return 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]

    def __call__(self, x):
        n, dev = x.shape[0], x.device
        pick = lambda p: (torch.rand(n, device=dev) < p).view(n, 1, 1, 1)
        x = torch.where(pick(0.5), x.flip(3), x)                    # horizontal flip
        x = torch.where(pick(0.5), x.flip(2), x)                    # vertical flip
        k = torch.randint(0, 4, (n,), device=dev).view(n, 1, 1, 1)
        x = torch.where(k == 0, x, torch.where(k == 1, x.rot90(1, (2, 3)),
                        torch.where(k == 2, x.rot90(2, (2, 3)), x.rot90(3, (2, 3)))))
        x = x.clamp(1e-6, 1) ** self._factor(n, self.gamma, dev)    # gamma
        x = x * self._factor(n, self.brightness, dev)               # brightness
        m = self._gray(x).mean(dim=(1, 2, 3), keepdim=True)
        x = (x - m) * self._factor(n, self.contrast, dev) + m       # contrast
        g = self._gray(x)
        x = (x - g) * self._factor(n, self.saturation, dev) + g     # saturation
        return (x.clamp(0, 1) - self.mean.to(dev)) / self.std.to(dev)


class GPULoader:
    """Wraps a DataLoader: moves each batch to the device and applies `augment` there."""
    def __init__(self, loader, augment):
        self.loader, self.augment = loader, augment
        self.dataset, self.generator = loader.dataset, loader.generator   # train_model uses these

    def __len__(self):
        return len(self.loader)

    def __iter__(self):
        for sample in self.loader:
            x = sample['image'].to(device, dtype=torch.float32, non_blocking=True)
            yield {'image': self.augment(x), 'label': sample['label']}

In [ ]:
train_dataset = FireRiskDataset(
    DB_DIR / 'train_reduced.csv', DB_DIR, maxSize=MAX_TRAIN_SAMPLES, preload=True,
    transform=transforms.Compose([
        RandomCrop(224), ToTensor()  
        ]))  # transform could be also = train_aug, but we want to use the GPUAugment class, which already normalizes and augments

val_dataset = FireRiskDataset(
    DB_DIR / 'val_reduced.csv', DB_DIR,
    transform=transforms.Compose([
        CenterCrop(224), ToTensor(), Normalize(PIXEL_MEAN, PIXEL_STD),
    ]))

test_dataset = FireRiskDataset(
    DB_DIR / 'test_reduced.csv', DB_DIR,
    transform=transforms.Compose([
        CenterCrop(224), ToTensor(), Normalize(PIXEL_MEAN, PIXEL_STD),
    ]))

for i in range(min(4, len(train_dataset))):
    sample = train_dataset[i]
    print(i, sample['image'].shape, int(sample['label']))

print(f'Training subset size: {len(train_dataset)}')

In [ ]:
def seed_worker(worker_id):
    # PyTorch assigns a worker seed; also seed NumPy and Python random.
    worker_seed = torch.initial_seed() % (2 ** 32) # ensure its within the range of a 32-bit integer
    np.random.seed(worker_seed)
    random.seed(worker_seed)

# Torch Generator used to seed the workers for reproducibility. 
# Generator() is a random number generator that can be used to control the randomness of operations in PyTorch
train_generator = torch.Generator().manual_seed(SEED) # Fixes the shuffle order 

# pin_memory speeds up CPU -> GPU copies; persistent_workers avoids re-spawning workers every epoch.
    # In windows, persistent_workers is ignored, as NUM_WORKERS is 0.
PIN_MEMORY = torch.cuda.is_available()
PERSISTENT = NUM_WORKERS > 0

# Batch size can be an hyperparameter in train (in val and test it doesnt change predictions)
train_dataloader = DataLoader(
    train_dataset, batch_size=64, shuffle=True, num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker, generator=train_generator,
    pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)

# In val and test we dont compute gradients so we can use a larger batch size.
val_dataloader = DataLoader(
    val_dataset, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)

test_dataloader = DataLoader(
    test_dataset, batch_size=128, shuffle=False, num_workers=NUM_WORKERS,
    worker_init_fn=seed_worker, pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)

In [ ]:
# ---- Architecture HYPERPARAMETERS ( to edit here, not inside the class) ----
INPUT_SIZE = 224                 # side of the square crop given to the network (see RandomCrop/CenterCrop)

# Initially, this is the reference setup. TODO: explore other architectures, hyperparams, methods and compare validations
CUSTOM_ARCH = dict(
    channels=(32, 64, 128, 256),        # nº of filters of each conv block (nº filters = nº blocks)
    kernel_size=3,               # conv kernel (stride 1)
    padding=1,                   # 0 shrinks the map; kernel_size // 2 keeps its size
    hidden=128,                  # out units of the first fully connected layer
    dropout=0.5,                 # dropout probability before the last layer (0 = off)
    batch_norm=True,            # BatchNorm after each conv
    convs_per_block=1,           # convs (Conv-BN-ReLU) before each MaxPool; 2 = VGG style
    head='flatten',              # 'flatten' = Flatten -> fc1 (reference); 'gap' = global average pooling -> fc1
)


class CustomNet(nn.Module):
    """CNN from scratch: N x 3 x S x S  ->  N x 7 raw logits (no softmax inside, softmax is implemented in loss function).

    [[Conv(k x k) -> (BatchNorm) -> ReLU] x convs_per_block -> MaxPool(2 x 2)] x len(channels) -> Flatten (or global
    average pooling if head='gap') -> fc1 -> ReLU -> (Dropout) -> fc2

    The input size of fc1 is autocomputed from input_size, so changing the input size, kernel,
    padding or number of blocks does not require editing the linear layer.
    """

    def __init__(self, channels=(6, 16, 32), kernel_size=5, padding=0, hidden=128,
                 dropout=0.0, batch_norm=False, input_size=INPUT_SIZE, num_classes=NUM_CLASSES,
                 convs_per_block=1, head='flatten'):
        super().__init__()
        if head not in ('flatten', 'gap'):
            raise ValueError("head must be 'flatten' or 'gap'.")

        blocks, in_channels, in_size = [], 3, input_size
        for out_channels in channels:
            for _ in range(convs_per_block):   # first conv: in_channels -> out_channels, the others: out_channels -> out_channels
                blocks.append(nn.Conv2d(in_channels, out_channels, kernel_size, padding=padding))

                # After each conv, before ReLu, to normalize the activations and gradients, improving training stability and speed.
                if batch_norm:
                    blocks.append(nn.BatchNorm2d(out_channels))
                blocks.append(nn.ReLU(inplace=True))

                in_size = in_size - kernel_size + 2 * padding + 1   # conv: size - k + 2p + 1
                # For the next conv, the input channels are the output channels of this one
                in_channels = out_channels
            # After the convs of the block, Pool
            blocks.append(nn.MaxPool2d(2))
            in_size //= 2                                           # pool: floor(size / 2)
        if in_size < 1:
            raise ValueError('The input is too small for this number of blocks.')

        # We call features to the sequence of Conv LAYERS created, it can be called in forward() to compute the feature maps of the input images.
        self.features = nn.Sequential(*blocks) 

        # FC1 Layer depends on input size and nº of blocks: compute number of features for the first FC layer
        # head='gap': global average pooling leaves one value per channel; 'flatten': the whole map (C*H*W)
        self.pool = nn.AdaptiveAvgPool2d(1) if head == 'gap' else nn.Identity()
        flat_features = in_channels if head == 'gap' else in_channels * in_size * in_size      # in_channels is the last one computed (last block out_channels)   

        # AFTER CONV BLOCKS:  Sequence of FC layers: Linear -> ReLU -> Dropout -> Linear (output)
            # It is the classifier part of the network, takes the flattened feature maps and produces the final class logits.            
        self.classifier = nn.Sequential(
            nn.Linear(flat_features, hidden), nn.ReLU(inplace=True), nn.Dropout(dropout), nn.Linear(hidden, num_classes))

        # He initialisation for ReLU convs
        for m in self.modules():  # For every layer in the network
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu') 
                nn.init.zeros_(m.bias)

    # Forward pass: x is a batch of images (N x 3 x Size x Size), returns N x 7 logits
    def forward(self, x):
        x = self.pool(self.features(x))   # N x C x H x W  (N x C x 1 x 1 with head='gap')
        # self.classifier receives a flattened input as its 1st layer is Linear
        x = x.flatten(1)             # N x (C*H*W) 
        return self.classifier(x)    # N x 7 logits

In [ ]:
from sklearn.metrics import f1_score, confusion_matrix, ConfusionMatrixDisplay

In [ ]:
def train_model(model, criterion, optimizer, scheduler,
                num_epochs=25, plot_confusion_matrix=True, *,
                loaders=None, amp=False, mixup_alpha=0.0, epoch_callback=None, verbose=True):
    """Run fixed-budget training and restore the best validation-macro-F1 model.

    Defaults reproduce the plain loop. Optional: `loaders` (dict 'train'/'val'; default global `dataloaders`),
    `amp` (fp16 autocast on CUDA), `mixup_alpha` (> 0: mixup), `epoch_callback(info)` (returns True to stop early),
    `verbose=False` (one compact line per epoch).
    """

    loaders = dataloaders if loaders is None else loaders
    use_amp = bool(amp) and device.type == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)

    if not isinstance(num_epochs, numbers.Integral) or num_epochs < 1:
        raise ValueError('num_epochs must be a positive integer.')
    if not isinstance(criterion, nn.CrossEntropyLoss) or criterion.reduction != 'mean':
        raise ValueError('Use CrossEntropyLoss with reduction="mean".')
    if any(len(loaders[phase].dataset) == 0 for phase in ('train', 'val')):
        raise ValueError('Training and validation datasets must be nonempty.')

    seed_everything()

    if loaders['train'].generator is not None:
        loaders['train'].generator.manual_seed(SEED)

    criterion = criterion.to(device)

    since = time.time()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_f1 = -float('inf')
    best_predictions = best_labels = None
    labels_order = np.arange(NUM_CLASSES)

    for epoch in range(num_epochs):
        if verbose:
            print(f'Epoch {epoch}/{num_epochs - 1}')
            print('-' * 10)

        for phase in ('train', 'val'):
            model.train(phase == 'train')
            running_loss = 0.0
            loss_denominator = 0.0
            phase_predictions, phase_labels = [], []

            for sample in loaders[phase]:
                inputs = sample['image'].to(device, dtype=torch.float32)
                labels = sample['label'].to(device, dtype=torch.long)

                mixing = phase == 'train' and mixup_alpha > 0
                if mixing:   # mixup: blend each image with a shuffled partner of the same batch
                    lam = float(np.random.beta(mixup_alpha, mixup_alpha))
                    perm = torch.randperm(inputs.shape[0], device=inputs.device)
                    inputs = lam * inputs + (1 - lam) * inputs[perm]

                if phase == 'train':
                    optimizer.zero_grad(set_to_none=True)
                with torch.set_grad_enabled(phase == 'train'):
                    with torch.autocast('cuda', dtype=torch.float16, enabled=use_amp):
                        logits = model(inputs)
                        if logits.shape != (labels.numel(), NUM_CLASSES):
                            raise ValueError('The network must return N x 7 logits.')
                        if mixing:
                            loss = lam * criterion(logits, labels) + (1 - lam) * criterion(logits, labels[perm])
                        else:
                            loss = criterion(logits, labels)
                    if not torch.isfinite(loss):
                        raise ValueError('Non-finite loss; check inputs and optimiser settings.')
                    if phase == 'train':
                        scaler.scale(loss).backward()
                        scaler.step(optimizer)
                        scaler.update()

                denominator = (labels.numel() if criterion.weight is None else
                               criterion.weight[labels].sum().item())
                running_loss += loss.item() * denominator
                loss_denominator += denominator
                phase_predictions.append(logits.detach().argmax(dim=1).cpu().numpy())
                phase_labels.append(labels.detach().cpu().numpy())   # original labels, also with mixup

            if phase == 'train' and scheduler is not None:
                scheduler.step()
            predictions = np.concatenate(phase_predictions)
            true_labels = np.concatenate(phase_labels)
            epoch_loss = running_loss / loss_denominator
            epoch_f1 = f1_score(true_labels, predictions, labels=labels_order,
                                average='macro', zero_division=0)
            if verbose:
                print(f'{phase} Loss: {epoch_loss:.4f} Macro F1-score: {epoch_f1:.4f}')
            if phase == 'train':
                train_loss, train_f1 = epoch_loss, epoch_f1
            if phase == 'val' and epoch_f1 > best_f1:
                best_f1 = epoch_f1
                best_model_wts = copy.deepcopy(model.state_dict())
                best_predictions = predictions.copy()
                best_labels = true_labels.copy()
        if verbose:
            print()
        else:
            print(f'ep {epoch + 1:2d}/{num_epochs}  train loss {train_loss:.3f} F1 {train_f1:.3f} | '
                  f'val loss {epoch_loss:.3f} F1 {epoch_f1:.3f}  (best {best_f1:.3f})')

        if epoch_callback is not None and epoch_callback(dict(
                epoch=epoch, train_loss=train_loss, train_f1=train_f1,
                val_loss=epoch_loss, val_f1=epoch_f1, best_f1=best_f1)):
            print(f'Pruned after epoch {epoch + 1}')
            break

    model.load_state_dict(best_model_wts)
    model.eval()

    elapsed = time.time() - since
    print(f'Training complete in {int(elapsed // 60)}m {int(elapsed % 60)}s')
    print(f'Best validation macro F1-score: {best_f1:.6f}')

    if plot_confusion_matrix:
        cm = confusion_matrix(best_labels, best_predictions, labels=labels_order,
                              normalize='true')
        display = ConfusionMatrixDisplay(100 * cm, display_labels=CLASS_NAMES)
        display.plot(xticks_rotation='vertical', values_format='.1f')
        plt.title('Row-normalised confusion matrix (%)')
        plt.tight_layout()
        plt.show()

    return model

In [ ]:
image_datasets = {'train': train_dataset, 'val': val_dataset}

# dataloaders = {'train': train_dataloader, 'val': val_dataloader}

gpu_augment = GPUAugment(PIXEL_MEAN, PIXEL_STD)
dataloaders = {'train': GPULoader(train_dataloader, gpu_augment), 'val': val_dataloader}

dataset_sizes = {phase: len(dataset) for phase, dataset in image_datasets.items()}
class_names = CLASS_NAMES.copy()

In [ ]:
SEARCH_DIR = WORK / 'search'          # small CSVs, versioned with git -> shared with the partner
MACHINE_ID = 1                       # <-- ONLY line that differs between the two computers (0 or 1)
N_MACHINES = 2                       # 1 = run every trial on this computer
SEARCH_EPOCHS = 15                   # budget per trial in phases 1-2
PRUNE_EPOCHS = (4, 8)                # median-stopping checks after these epochs
MIN_TRIALS_FOR_PRUNING = 4           # need this many finished trials of the phase before pruning
SEARCH_AMP = device.type == 'cuda'   # mixed precision: ~2x faster on the GPU
N_PHASE1, N_PHASE2 = 24, 16          # number of random trials in phase 1 / phase 2 (in total, over both computers)
N_FINALISTS, FINAL_EPOCHS = 3, 40    # phase 3: how many configs are retrained, and for how many epochs

# Fixed starting point: every search trial changes some of these values (the rest stay as they are here)
BASE_CFG = dict(base_channels=32, n_blocks=4,        # filters of block i = min(base_channels * 2**i, 512); number of blocks
                convs_per_block=1, head='gap',       # convs per block (VGG style); 'gap' = global avg pooling, 'flatten'
                hidden=128, dropout=0.3,             # fc1 units; dropout before the last layer
                optimizer='adamw', lr=1e-3, weight_decay=0.05, batch_size=64,
                warmup_epochs=1,                     # linear lr warm-up, then cosine decay
                label_smoothing=0.0, class_weights='none',   # class_weights: 'none' | 'sqrt_inv'
                mixup_alpha=0.0, aug='light')        # mixup strength (0 = off); GPU augmentation preset

SEARCH_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR = WORK / 'outputs' / 'Lab_4_2026_2027'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('SEARCH_DIR =', SEARCH_DIR, '| OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
import json, math, statistics

# GPU augmentation presets ('light' == the GPUAugment defaults)
AUG_PRESETS = {'light': dict(gamma=0.1, brightness=0.05, contrast=0.05, saturation=0.05),
               'strong': dict(gamma=0.2, brightness=0.15, contrast=0.15, saturation=0.15)}

def class_weight_tensor(kind):
    """None, or inverse-sqrt-frequency class weights (mean 1) computed on the training labels."""
    if kind == 'none':
        return None
    counts = np.bincount(train_dataset.dataset['label'], minlength=NUM_CLASSES)[:NUM_CLASSES]
    w = 1 / np.sqrt(np.maximum(counts, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

def build_run(cfg, epochs):
    """Model, loss, optimiser, scheduler and loaders for one configuration."""
    seed_everything()
    channels = tuple(min(cfg['base_channels'] * 2 ** i, 512) for i in range(cfg['n_blocks']))
    model = CustomNet(channels=channels, kernel_size=3, padding=1, hidden=cfg['hidden'], dropout=cfg['dropout'],
                      batch_norm=True, convs_per_block=cfg['convs_per_block'], head=cfg['head'],
                      input_size=INPUT_SIZE).to(device)
    criterion = nn.CrossEntropyLoss(weight=class_weight_tensor(cfg['class_weights']),
                                    label_smoothing=cfg['label_smoothing'])
    if cfg['optimizer'] == 'adamw':
        optimizer = optim.AdamW(model.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    else:
        optimizer = optim.SGD(model.parameters(), lr=cfg['lr'], momentum=0.9, nesterov=True,
                              weight_decay=cfg['weight_decay'])
    w = cfg['warmup_epochs']   # stepped once per epoch: linear warm-up, then cosine decay to 0
    scheduler = lr_scheduler.LambdaLR(optimizer, lambda e: (e + 1) / (w + 1) if e < w else
                                      0.5 * (1 + math.cos(math.pi * (e - w) / max(1, epochs - w))))
    train_loader = DataLoader(train_dataset, batch_size=cfg['batch_size'], shuffle=True, num_workers=NUM_WORKERS,
                              worker_init_fn=seed_worker, generator=torch.Generator().manual_seed(SEED),
                              pin_memory=PIN_MEMORY, persistent_workers=PERSISTENT)
    loaders = {'train': GPULoader(train_loader, GPUAugment(PIXEL_MEAN, PIXEL_STD, **AUG_PRESETS[cfg['aug']])),
               'val': val_dataloader}
    return model, criterion, optimizer, scheduler, loaders

# ---- search spaces (deterministic: same list on both computers) ----
def _loguniform(rng, lo, hi):
    return math.exp(rng.uniform(math.log(lo), math.log(hi)))

def _round3(v):
    return float(f'{v:.3g}')   # 3 significant digits, keeps the CSV readable

def sample_phase1(n, seed=SEED):
    """Primary hyperparameters: optimiser, lr, width, depth, convs per block, head."""
    rng, configs = random.Random(seed + 1), []
    for i in range(n):
        cfg = dict(BASE_CFG)
        cfg['optimizer'] = rng.choice(['adamw', 'sgd'])
        if cfg['optimizer'] == 'adamw':
            cfg['lr'], cfg['weight_decay'] = _round3(_loguniform(rng, 2e-4, 3e-3)), 0.05
        else:
            cfg['lr'], cfg['weight_decay'] = _round3(_loguniform(rng, 5e-3, 1e-1)), 5e-4
        cfg['base_channels'] = rng.choice([24, 32, 48])
        cfg['n_blocks'] = rng.choice([4, 5])
        cfg['convs_per_block'] = rng.choice([1, 2])
        cfg['head'] = rng.choice(['gap', 'flatten'])
        cfg['trial_id'] = f'p1_{i:02d}'
        configs.append(cfg)
    return configs

def sample_phase2(best, n, seed=SEED):
    """Secondary hyperparameters around `best` (keeps its architecture and optimiser)."""
    rng, configs = random.Random(seed + 2), []
    for i in range(n):
        cfg = dict(best)
        cfg['lr'] = _round3(best['lr'] * 2 ** rng.uniform(-1, 1))
        lo, hi = (5e-3, 0.2) if best['optimizer'] == 'adamw' else (1e-4, 2e-3)
        cfg['weight_decay'] = _round3(_loguniform(rng, lo, hi))
        cfg['dropout'] = rng.choice([0.0, 0.2, 0.3, 0.5])
        cfg['hidden'] = rng.choice([128, 256])
        cfg['batch_size'] = rng.choice([32, 64, 128])
        cfg['label_smoothing'] = rng.choice([0.0, 0.1])
        cfg['class_weights'] = rng.choice(['none', 'sqrt_inv'])
        cfg['mixup_alpha'] = rng.choice([0.0, 0.2, 0.4])
        cfg['aug'] = rng.choice(['light', 'strong'])
        cfg['trial_id'], cfg['parent'] = f'p2_{i:02d}', best['trial_id']
        configs.append(cfg)
    return configs

# ---- results store: one CSV per machine and phase in SEARCH_DIR (merged when reading) ----
RESULT_COLS = ['trial_id', 'phase', 'machine', 'status', 'best_val_f1', 'best_epoch', 'epochs_run',
               'minutes', 'params', 'gpu', 'val_f1_history', 'cfg']

def read_results(phase=None):
    """All trials of every machine (optionally one phase), best validation macro F1 first."""
    files = sorted(SEARCH_DIR.glob('results_*.csv'))
    df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True) if files else pd.DataFrame(columns=RESULT_COLS)
    if phase is not None:
        df = df[df['phase'] == phase]
    return df.sort_values('best_val_f1', ascending=False).reset_index(drop=True)

def show_leaderboard(phase=None, top=10):
    df = read_results(phase).head(top)
    if df.empty:
        print('No results yet.')
        return
    cfgs = pd.DataFrame([json.loads(c) for c in df['cfg']]).drop(columns=['trial_id'], errors='ignore')
    out = pd.concat([df.drop(columns=['val_f1_history', 'cfg']).reset_index(drop=True), cfgs], axis=1)
    out['best_val_f1'] = out['best_val_f1'].round(4)
    display(out)

def best_config(phase=None, rank=0):
    """cfg (with trial_id) of the given rank of the merged leaderboard (done or pruned trials)."""
    df = read_results(phase)
    row = df[df['status'].isin(['done', 'pruned'])].iloc[rank]
    return {**json.loads(row['cfg']), 'trial_id': row['trial_id']}

def append_result(row, phase):
    """Append one row to this machine's CSV right after the trial (resume-safe)."""
    path = SEARCH_DIR / f'results_{phase}_m{MACHINE_ID}.csv'
    pd.DataFrame([row], columns=RESULT_COLS).to_csv(path, mode='a', header=not path.exists(), index=False)

def _row(cfg, phase, status, history, params=0, minutes=0.0):
    return dict(trial_id=cfg['trial_id'], phase=phase, machine=MACHINE_ID, status=status,
                best_val_f1=max(history, default=0.0), best_epoch=int(np.argmax(history)) + 1 if history else 0,
                epochs_run=len(history), minutes=round(minutes, 2), params=params,
                gpu=torch.cuda.get_device_name(0) if device.type == 'cuda' else 'cpu',
                val_f1_history=json.dumps([round(float(h), 4) for h in history]), cfg=json.dumps(cfg))

def run_trial(cfg, phase, epochs, prune=True):
    """Train one config. Median stopping: after epochs in PRUNE_EPOCHS stop if the trial's best val F1 so far
    is below the median of what the finished trials of the same phase had reached by the same epoch."""
    since = time.time()
    model, criterion, optimizer, scheduler, loaders = build_run(cfg, epochs)
    history = []
    finished = [json.loads(h) for h in read_results(phase)['val_f1_history']]

    def cb(info):
        history.append(info['val_f1'])
        e = len(history)
        if not prune or e not in PRUNE_EPOCHS:
            return False
        reference = [max(h[:e]) for h in finished if len(h) >= e]
        if len(reference) >= MIN_TRIALS_FOR_PRUNING and max(history) < statistics.median(reference):
            print(f'Pruning: best val F1 {max(history):.3f} after {e} epochs < median {statistics.median(reference):.3f} '
                  f'of {len(reference)} finished trials')
            return True
        return False

    model = train_model(model, criterion, optimizer, scheduler, num_epochs=epochs, plot_confusion_matrix=False,
                        loaders=loaders, amp=SEARCH_AMP, mixup_alpha=cfg['mixup_alpha'], epoch_callback=cb, verbose=False)
    status = 'pruned' if len(history) < epochs else 'done'
    return model, _row(cfg, phase, status, history, sum(p.numel() for p in model.parameters()),
                       (time.time() - since) / 60)

def run_phase(phase, configs, epochs=SEARCH_EPOCHS, prune=True):
    """Run this machine's share of `configs` (index % N_MACHINES == MACHINE_ID), skipping trials already stored."""
    import gc
    mine = [c for i, c in enumerate(configs) if i % N_MACHINES == MACHINE_ID]
    done_ids = set(read_results()['trial_id'])
    for cfg in mine:
        if cfg['trial_id'] in done_ids:
            continue
        print(f"\n=== {cfg['trial_id']} (machine {MACHINE_ID}) {cfg}")
        try:
            model, row = run_trial(cfg, phase, epochs, prune)
        except torch.cuda.OutOfMemoryError:
            print('CUDA out of memory -> trial skipped')
            model, row = None, _row(cfg, phase, 'oom', [])
        append_result(row, phase)
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
    show_leaderboard(phase)

In [ ]:
# ---- Phase 4: targeted experiments on p2_07 (the most stable phase-3 finalist), one change at a time ----
import copy, gc

def class_weight_tensor(kind):
    """'none' | 'sqrt_inv' (1/sqrt(n)) | 'inv' (1/n); normalised to mean 1. Overrides the earlier definition."""
    if kind == 'none':
        return None
    counts = np.bincount(train_dataset.dataset['label'], minlength=NUM_CLASSES)[:NUM_CLASSES]
    counts = np.maximum(counts, 1).astype(np.float64)
    w = 1 / np.sqrt(counts) if kind == 'sqrt_inv' else 1 / counts
    return torch.tensor(w / w.mean(), dtype=torch.float32, device=device)

# p2_07 config, copied from results_p3_m1.csv (so no previous CSV is needed here)
BASE_P4 = dict(base_channels=32, n_blocks=4, convs_per_block=1, head='gap', hidden=256, dropout=0.3,
               optimizer='adamw', lr=0.000317, weight_decay=0.0464, batch_size=128, warmup_epochs=1,
               label_smoothing=0.0, class_weights='sqrt_inv', mixup_alpha=0.0, aug='strong')
REFERENCE = {'experimento': 'p3_p2_07 (fase 3, Colab T4)', 'val_last10': 0.4561, 'val_max': 0.4645,
             'train_last10': 0.4503, 'train_loss_final': 1.258}

P4_EXPERIMENTS = {
    'p4_A_inv':        dict(class_weights='inv'),
    'p4_B_wide48x2':   dict(base_channels=48, convs_per_block=2),
    'p4_C_wide_noreg': dict(base_channels=48, convs_per_block=2, dropout=0.0, aug='light'),
}
RUN_ONLY = list(P4_EXPERIMENTS)   # all three in one go (~1 h on a T4); or e.g. ['p4_A_inv']
P4_EPOCHS = 40

_original_train_model = train_model
epoch_log = []
def train_model(*args, epoch_callback=None, **kwargs):
    def logging_callback(info):
        epoch_log.append({'epoch': info['epoch'] + 1, 'train_f1': info['train_f1'], 'val_f1': info['val_f1'],
                          'train_loss': info['train_loss'], 'val_loss': info['val_loss']})
        return epoch_callback(info) if epoch_callback is not None else False
    return _original_train_model(*args, epoch_callback=logging_callback, **kwargs)

try:
    for tid in RUN_ONLY:
        cfg = {**copy.deepcopy(BASE_P4), **P4_EXPERIMENTS[tid], 'trial_id': tid, 'parent': 'p2_07'}
        ckpt = OUTPUT_DIR / 'search' / f'{tid}.pt'
        if tid in set(read_results('p4')['trial_id']) and ckpt.exists():
            print(f'{tid}: already trained, skipping')
            continue
        print(f'\n=== {tid} {P4_EXPERIMENTS[tid]}')
        epoch_log.clear()
        try:
            model, row = run_trial(cfg, 'p4', P4_EPOCHS, prune=False)
        except torch.cuda.OutOfMemoryError:
            print('CUDA out of memory -> rerun this experiment with batch_size=64')
            torch.cuda.empty_cache()
            continue
        ckpt.parent.mkdir(parents=True, exist_ok=True)
        torch.save({'state_dict': model.state_dict(), 'cfg': cfg}, ckpt)
        pd.DataFrame(epoch_log).to_csv(OUTPUT_DIR / 'search' / f'{tid}_epochs.csv', index=False)
        append_result(row, 'p4')
        del model
        gc.collect()
        if device.type == 'cuda':
            torch.cuda.empty_cache()
finally:
    train_model = _original_train_model

In [ ]:
# ---- Summary: mean of the last 10 epochs (robust metric) against the p2_07 reference ----
rows = [REFERENCE]
for tid in P4_EXPERIMENTS:
    f = OUTPUT_DIR / 'search' / f'{tid}_epochs.csv'
    if f.exists():
        d = pd.read_csv(f)
        rows.append({'experimento': tid, 'val_last10': d['val_f1'].tail(10).mean(), 'val_max': d['val_f1'].max(),
                     'train_last10': d['train_f1'].tail(10).mean(), 'train_loss_final': d['train_loss'].iloc[-1]})
summary = pd.DataFrame(rows).round(4)
summary['vs_p2_07'] = (summary['val_last10'] - REFERENCE['val_last10']).round(4)
summary.to_csv(OUTPUT_DIR / 'search' / 'p4_summary.csv', index=False)
display(summary)
print('Files to download (Output tab):', sorted(str(p.relative_to(WORK)) for p in (WORK / 'outputs').rglob('*.csv')))